# Spark SQL


Показать отбор, вычисляемые поля, группировку и вложенные запросы на небольших результатах.

Таблицы примеров доступны через трёхчастное имя `lakehouse.raw...`. SQL в ячейках записан строчными буквами.

Ограничения: Когда запрос должен быть прозрачен аналитикам, переносим между SQL-системами или удобен для проверки. Не путайте ограничение выдачи с ограничением чтения: изучайте план и фильтруйте по столбцам, имеющим смысл для источника.


## Основные понятия
SQL описывает намерение декларативно. Отбор строк задаётся `where`, группировка объединяет строки по ключу, а общее табличное выражение именует промежуточный шаг.


In [ ]:
from lakehouse_spark import SparkSessionContext
from pyspark.sql import functions as F

## Примеры


**1. Выбор и вычисляемые поля**

Выберите только нужные поля и ограничьте диапазон длительности.



In [ ]:
with SparkSessionContext('example-sql-selection') as spark:
    spark.sql("""
        select match_id, round(duration / 60.0, 1) as duration_minutes,
               radiant_score + dire_score as total_kills, radiant_win
        from lakehouse.raw.pg__matches
        where duration between 600 and 7200
        order by total_kills desc
        limit 10
    """).show(truncate=False)


**2. Группировка и пропуски**

Сравните число строк, среднее и медиану по признаку победы.



In [ ]:
with SparkSessionContext('example-sql-aggregation') as spark:
    spark.sql("""
        select radiant_win, count(*) as matches,
               round(avg(duration), 1) as average_duration,
               percentile_approx(duration, 0.5) as median_duration
        from lakehouse.raw.pg__matches
        where duration is not null and duration > 0
        group by radiant_win
        order by radiant_win
    """).show(truncate=False)

**3. Общее табличное выражение**

Сначала посчитайте статистику по героям, затем добавьте название героя.



In [ ]:
with SparkSessionContext('example-sql-cte') as spark:
    spark.sql("""
        with hero_stats as (
            select hero_id, count(*) as games, round(avg(kills), 2) as average_kills
            from lakehouse.raw.pg__match_players
            where hero_id is not null
            group by hero_id
        )
        select h.localized_name, s.games, s.average_kills
        from hero_stats s
        left join lakehouse.raw.pg__heroes h using (hero_id)
        order by s.games desc
        limit 15
    """).show(truncate=False)

**4. Вложенный запрос**

Отделите расчёт условного показателя от сортировки итоговой выдачи.



In [ ]:
with SparkSessionContext('example-sql-subquery') as spark:
    spark.sql("""
        select radiant_win, matches, long_matches
        from (
            select radiant_win, count(*) as matches,
                   sum(case when duration >= 3600 then 1 else 0 end) as long_matches
            from lakehouse.raw.pg__matches
            group by radiant_win
        ) summary
        order by radiant_win
    """).show(truncate=False)

**5. Проверка типов**

Посмотрите типы и несколько значений, прежде чем использовать арифметику.



In [ ]:
with SparkSessionContext('example-sql-types') as spark:
    spark.sql("describe lakehouse.raw.pg__matches").show(100, truncate=False)